## 1. Importazioni e Setup / Imports & Setup

**In italiano:** Importiamo tutte le librerie necessarie per l'esecuzione standalone dell'agente. Usiamo `langchain_google_genai` per interagire con l'LLM di Gemini, `DuckDuckGoSearchResults` per la ricerca sul web, `langgraph` per definire il flusso a stati dell'agente, e `transformers` con `torch` per eseguire localmente il modello di Natural Language Inference (NLI). Carichiamo anche il file `.env` dal progetto per le chiavi API.

**In English:** Import all the libraries required for the standalone execution of the agent. We use `langchain_google_genai` to interact with Gemini's LLM, `DuckDuckGoSearchResults` for web searching, `langgraph` to define the agent's state flow, and `transformers` with `torch` to run the Natural Language Inference (NLI) model locally. We also load the `.env` file from the project for API keys.


In [ ]:
import os
import re
import sys
from typing import Any, TypedDict

import torch
from dotenv import load_dotenv
from IPython.display import Image, display
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_google_genai import ChatGoogleGenerativeAI
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Carica il file .env per ottenere la GOOGLE_API_KEY / GEMINI_API_KEY
# Load the .env file to retrieve the GOOGLE_API_KEY / GEMINI_API_KEY
if os.path.exists("../.env"):
    load_dotenv("../.env")
elif os.path.exists("../backend/.env"):
    load_dotenv("../backend/.env")
else:
    load_dotenv()


## 2. Definizione degli Stati / State Definition

**In italiano:** Definiamo tre strutture dati per gestire lo stato dell'agente:
* **InputState**: Riceve il claim o la notizia originale inserita dall'utente.
* **HiddenState**: Memorizza tutti i passaggi intermedi della computazione dell'agente, inclusi la query di ricerca raffinata, le evidenze grezze estratte, i link alle fonti, il verdetto NLI con la sua confidenza tecnica e la spiegazione testuale.
* **OutputState**: Rappresenta il risultato finale che viene esposto alla fine del workflow.

**In English:** We define three data structures to manage the agent's state:
* **InputState**: Receives the original user claim or news.
* **HiddenState**: Stores all intermediate computation steps, including the refined search query, raw extracted evidence, links to sources, the NLI verdict with its technical confidence, and the final textual explanation.
* **OutputState**: Represents the final results exposed at the end of the workflow.


In [ ]:
class InputState(TypedDict):
    query: str


class HiddenState(TypedDict):
    query: str
    search_query: str
    premise: str
    researches: str
    sources: list[str]
    retrieved_docs: list[str]
    nli_label: str
    confidence: float
    nli_model_label: str
    nli_class_id: int
    verdict_probabilities: dict[str, float]
    motivation: str
    response: str


class OutputState(TypedDict):
    query: str
    search_query: str
    researches: str
    sources: list[str]
    retrieved_docs: list[str]
    nli_label: str
    confidence: float
    nli_model_label: str
    nli_class_id: int
    verdict_probabilities: dict[str, float]
    motivation: str
    response: str


## 3. Inizializzazione dei Modelli / Models Initialization

**In italiano:** Inizializziamo i modelli necessari:
- **LLM**: Usiamo il modello generativo configurato (es. `gemma-4-31b-it`) via Gemini API. Se la chiave API non è configurata, i nodi LLM useranno un meccanismo di fallback locale.
- **NLI Classifier**: Carichiamo il modello DeBERTa fine-tuned per il dataset FEVER. Il sistema cerca prima il modello salvato localmente nella cartella `../fever-nli-deberta` (cartella radice del progetto rispetto a questo notebook). Se non è presente, effettua il download del modello di base cross-encoder da Hugging Face.

**In English:** Initialize the required models:
- **LLM**: We use the configured generative model (e.g., `gemma-4-31b-it`) via Gemini API. If the API key is not configured, LLM nodes will use a local fallback mechanism.
- **NLI Classifier**: We load the DeBERTa model fine-tuned on the FEVER dataset. The system first checks for the locally saved model under `../fever-nli-deberta` (project root folder relative to this notebook). If it is not found, it downloads the base cross-encoder model from Hugging Face.


In [ ]:
# Inizializziamo le variabili globali del modello
# Initialize global model variables
llm = None
tokenizer = None
nli_model = None
device = None

# Configura l'LLM
api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
if not api_key:
    print("WARNING: GOOGLE_API_KEY o GEMINI_API_KEY non trovata. Verranno usati i fallback locali.")
else:
    model_name = os.getenv("GEMINI_MODEL", "gemma-4-31b-it")
    print(f"Uso modello generativo LLM: {model_name}")
    llm = ChatGoogleGenerativeAI(model=model_name, api_key=api_key)

# Risolvi il percorso per il modello NLI locale
model_path = "../fever-nli-deberta"
if not os.path.exists(model_path):
    print(f"Modello locale non trovato in {model_path}. Uso il modello base da HuggingFace come fallback.")
    model_path = "cross-encoder/nli-deberta-v3-large"
else:
    print(f"Caricamento modello NLI locale da: {model_path}")

# Caricamento Tokenizer e Modello NLI
tokenizer = AutoTokenizer.from_pretrained(model_path)
nli_model = AutoModelForSequenceClassification.from_pretrained(model_path)
nli_model.eval()

# Sposta il modello sul device corretto (GPU o CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nli_model.to(device)

print(f"Inizializzazione completata. Modello NLI caricato su: {device}")


## 4. Definizione dei Nodi (Architettura RAG-NLI) / Nodes Definition

**In italiano:** In questa sezione sono definiti tutti i nodi e le relative funzioni di supporto del grafo dell'agente. I nodi implementano i passi principali dell'elaborazione:
- `refine_query_node`: Genera una query neutrale a partire dalla notizia dell'utente (con fallback locale a parole chiave).
- `search_node`: Ricerca su internet tramite DuckDuckGo, formattando i risultati ed estraendo i link delle fonti in modo sicuro.
- `nli_classification_node`: Converte premise ed hypothesis in input per DeBERTa, esegue l'inferenza e mappa la classe predetta nel verdetto FEVER (`SUPPORTS`, `REFUTES`, `NOT ENOUGH INFO`), calcolando anche la distribuzione di probabilità.
- `generate_motivation_node`: Utilizza l'LLM per generare un testo esplicativo in italiano con le metriche tecniche e i link delle fonti.

**In English:** This section defines all the nodes and their helper functions for the agent's graph. The nodes implement the core processing steps:
- `refine_query_node`: Generates an optimal search query from the user claim (with local keyword extraction fallback).
- `search_node`: Searches the web using DuckDuckGo, formatting results and extracting source links securely.
- `nli_classification_node`: Converts premise and hypothesis to DeBERTa input formats, runs inference, and maps the predicted class to the FEVER labels (`SUPPORTS`, `REFUTES`, `NOT ENOUGH INFO`), while calculating probability distributions.
- `generate_motivation_node`: Invokes the LLM to generate a narrative explanation in Italian, appending technical metrics and source URLs.


In [ ]:
ITALIAN_STOPWORDS = {
    "a", "ad", "al", "alla", "allo", "anche", "che", "chi", "con", "da", "dal",
    "dalla", "dello", "dei", "del", "della", "di", "e", "ed", "gli", "ha",
    "hanno", "i", "il", "in", "la", "le", "lo", "ma", "nel", "nella", "non",
    "o", "per", "piu", "più", "quale", "quali", "se", "si", "sia", "sono",
    "su", "tra", "un", "una",
}


def _format_search_result(result: dict[str, str], index: int) -> str:
    """Formatta un singolo risultato di ricerca in markdown."""
    title = result.get("title") or "Senza titolo"
    snippet = result.get("snippet") or "Snippet non disponibile."
    link = result.get("link") or "Link non disponibile."
    return f"[{index}] {title}\nSnippet: {snippet}\nLink: {link}"


def _normalize_search_results(results: Any) -> list[dict[str, str]]:
    """Normalizza i risultati grezzi di DuckDuckGo in un formato standard."""
    if not isinstance(results, list):
        return []

    normalized_results: list[dict[str, str]] = []
    for item in results:
        if not isinstance(item, dict):
            continue

        normalized_results.append(
            {
                "title": str(item.get("title", "")).strip(),
                "snippet": str(item.get("snippet", "")).strip(),
                "link": str(item.get("link", "")).strip(),
            }
        )

    return normalized_results


def _extract_sources(results: list[dict[str, str]]) -> list[str]:
    """Estrae i link univoci delle fonti dai risultati normalizzati."""
    seen: set[str] = set()
    sources: list[str] = []

    for result in results:
        link = result.get("link", "").strip()
        if not link or link in seen:
            continue
        seen.add(link)
        sources.append(link)

    return sources


def _extract_llm_text(response: Any) -> str:
    """Estrae e pulisce il testo di risposta restituito dall'LLM."""
    content = response.content
    if isinstance(content, list):
        content = "".join(
            part.get("text", "") if isinstance(part, dict) else str(part)
            for part in content
            if not isinstance(part, dict) or part.get("type") != "thinking"
        )
    return str(content).strip()


def _format_confidence_percentage(value: float) -> str:
    """Converte un valore decimale di confidenza in percentuale stringa."""
    return f"{round(value * 100, 1)}%"


def _format_sources_for_prompt(sources: list[str]) -> str:
    """Formatta i link delle fonti per l'inserimento nel prompt."""
    if not sources:
        return "- Nessuna fonte strutturata disponibile."
    return "\n".join(f"- {source}" for source in sources)


def _build_verdict_probabilities(probabilities: torch.Tensor) -> dict[str, float]:
    """Raggruppa le probabilità grezze del modello nei verdetti FEVER."""
    verdict_probabilities = {
        "SUPPORTS": 0.0,
        "REFUTES": 0.0,
        "NOT ENOUGH INFO": 0.0,
    }

    for class_id, probability in enumerate(probabilities[0].tolist()):
        model_label = _resolve_model_label(class_id)
        verdict = _map_model_label_to_verdict(model_label)
        verdict_probabilities[verdict] += float(probability)

    return verdict_probabilities


def _format_verdict_probabilities(verdict_probabilities: dict[str, float]) -> str:
    """Formatta le probabilità aggregate per verdetto in formato leggibile."""
    verdict_order = ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]
    return ", ".join(
        f"{verdict}={_format_confidence_percentage(verdict_probabilities.get(verdict, 0.0))}"
        for verdict in verdict_order
    )


def _build_technical_summary(state: HiddenState) -> str:
    """Costruisce la stringa con le informazioni tecniche sul verdetto."""
    verdict = state["nli_label"]
    confidence = _format_confidence_percentage(state["confidence"])
    model_label = state.get("nli_model_label", "n/d")
    class_id = state.get("nli_class_id", "n/d")
    verdict_probabilities = state.get("verdict_probabilities", {})
    probabilities_text = _format_verdict_probabilities(verdict_probabilities)

    return (
        f"**Verdetto tecnico:** {verdict}. "
        f"**Confidenza NLI:** {confidence}. "
        f"**Classe raw del modello:** {model_label} (id={class_id}). "
        f"**Distribuzione:** {probabilities_text}."
    )


def _prepend_technical_summary(text: str, state: HiddenState) -> str:
    """Prepone il riepilogo tecnico al testo esplicativo generato."""
    summary = _build_technical_summary(state)
    if text.startswith(summary):
        return text
    return summary + "\n\n" + text.strip()


def _fallback_search_query(query: str) -> str:
    """Estrae parole chiave in locale se l'LLM non è configurato."""
    words = re.findall(r"\w+", query.lower())
    keywords = [word for word in words if len(word) > 2 and word not in ITALIAN_STOPWORDS]
    selected_words = keywords[:6] or words[:6]
    return " ".join(selected_words).strip() or query.strip()


def _fallback_motivation(state: HiddenState) -> str:
    """Genera la spiegazione finale in locale in caso di LLM offline."""
    verdict = state["nli_label"]
    verdict_text = {
        "SUPPORTS": "le evidenze recuperate sono coerenti con il claim",
        "REFUTES": "le evidenze recuperate contraddicono il claim",
        "NOT ENOUGH INFO": "le evidenze recuperate non bastano per una conclusione solida",
    }.get(verdict, "il sistema non ha prodotto un verdetto interpretabile")

    evidence_lines = state.get("retrieved_docs", [])[:2]
    evidence_text = "\n\n".join(evidence_lines) if evidence_lines else state["researches"]

    conclusion = {
        "SUPPORTS": "Nel complesso, la notizia risulta supportata dalle fonti trovate.",
        "REFUTES": "Nel complesso, la notizia risulta smentita dalle fonti trovate.",
        "NOT ENOUGH INFO": "Nel complesso, le fonti trovate non bastano per confermare o smentire la notizia.",
    }.get(verdict, "Nel complesso, serve una verifica manuale aggiuntiva.")

    return (
        "**Oggetto: Verifica della notizia**\n\n"
        f"{_build_technical_summary(state)}\n\n"
        f"Il verdetto del sistema e' **{verdict}** con una confidenza del {_format_confidence_percentage(state['confidence'])}. "
        f"In termini pratici, significa che {verdict_text}.\n\n"
        "Le evidenze principali considerate dal sistema sono le seguenti:\n\n"
        f"{evidence_text}\n\n"
        "Questa spiegazione e' stata generata con il fallback locale perche' il modello LLM non e' configurato.\n\n"
        f"{conclusion}"
    )


def _map_model_label_to_verdict(model_label: str) -> str:
    """Mappa l'etichetta del modello nel dominio FEVER (SUPPORTS, REFUTES, NOT ENOUGH INFO)."""
    normalized_label = model_label.strip().lower().replace(" ", "_")
    verdict_map = {
        "entailment": "SUPPORTS",
        "supports": "SUPPORTS",
        "support": "SUPPORTS",
        "contradiction": "REFUTES",
        "refutes": "REFUTES",
        "refute": "REFUTES",
        "neutral": "NOT ENOUGH INFO",
        "not_enough_info": "NOT ENOUGH INFO",
        "nei": "NOT ENOUGH INFO",
    }

    if normalized_label not in verdict_map:
        raise ValueError(f"Etichetta NLI del modello non riconosciuta: {model_label}")

    return verdict_map[normalized_label]


def _resolve_model_label(predicted_class_id: int) -> str:
    """Risolve l'id classe numerica nella corrispondente stringa di label usando la config."""
    if nli_model is None:
        raise RuntimeError("Modello NLI non inizializzato.")

    id2label = getattr(nli_model.config, "id2label", {}) or {}
    model_label = id2label.get(predicted_class_id, id2label.get(str(predicted_class_id), ""))

    if model_label:
        return str(model_label)

    label2id = getattr(nli_model.config, "label2id", {}) or {}
    for label, class_id in label2id.items():
        if class_id == predicted_class_id:
            return str(label)

    raise ValueError(f"Impossibile risolvere la label NLI per class_id={predicted_class_id}")


def refine_query_node(state: HiddenState) -> dict[str, Any]:
    """Estrae le parole chiave essenziali o formula una query neutrale."""
    prompt = f"""
    Genera una query di ricerca neutrale per Google/DuckDuckGo a partire da questo testo: \"{state['query']}\"
    Mantieni solo i termini informativi essenziali, come nomi propri, luoghi, date, enti o evento principale.
    Non aggiungere parole che suggeriscano un esito, una verifica o un giudizio, come "vero", "falso", "bufala", "smentita" o simili.
    Se utile, riformula in modo breve e generico senza cambiare il significato.
    Lunghezza massima: 5-6 parole.
    Rispondi solo con la query, senza virgolette, formattazione o testo aggiuntivo.
    """
    if llm is None:
        search_query = _fallback_search_query(state["query"])
    else:
        res = llm.invoke(prompt)
        search_query = _extract_llm_text(res)

    print(f"[Refine Node] Query di ricerca generata: {search_query}")
    return {"search_query": search_query}


def search_node(state: HiddenState) -> dict[str, Any]:
    """Ricerca evidenze su internet e gestisce eventuali errori in modo robusto."""
    retrieved_docs: list[str] = []
    sources: list[str] = []
    researches = "Nessun risultato trovato a causa di un errore di connessione."

    try:
        search = DuckDuckGoSearchResults(output_format="list", num_results=5)
        raw_results = search.invoke(state["search_query"])
        normalized_results = _normalize_search_results(raw_results)
        retrieved_docs = [
            _format_search_result(result, index)
            for index, result in enumerate(normalized_results, start=1)
        ]
        premise = "\n\n".join([res['snippet'] for res in normalized_results]) if retrieved_docs else "Nessun risultato trovato."
        researches = "\n\n".join(retrieved_docs) if retrieved_docs else "Nessun risultato trovato."
        sources = _extract_sources(normalized_results)
    except Exception as e:
        print(f"[Search Node] Errore durante la ricerca sul web: {e}")

    print(
        f"[Search Node] Risultati estratti: {len(retrieved_docs)} documenti, "
        f"{len(sources)} fonti."
    )
    return {
        "premise": premise,
        "researches": researches,
        "retrieved_docs": retrieved_docs,
        "sources": sources,
    }


def nli_classification_node(state: HiddenState) -> dict[str, Any]:
    """Confronta l'evidenza con il claim originale usando il classificatore DeBERTa NLI."""
    premise = state["researches"]
    hypothesis = state["query"]

    if tokenizer is None or nli_model is None:
        raise RuntimeError(
            "Modello NLI o Tokenizer non inizializzati. Assicurati di caricare i modelli prima di avviare il grafo."
        )

    inputs = tokenizer(
        premise,
        hypothesis,
        padding="max_length",
        truncation=True,
        max_length=256,
        return_tensors="pt",
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = nli_model(**inputs)

    logits = outputs.logits
    probs = torch.nn.functional.softmax(logits, dim=-1)
    predicted_class_id = logits.argmax().item()
    confidence = probs[0, predicted_class_id].item()
    model_label = _resolve_model_label(predicted_class_id)
    label = _map_model_label_to_verdict(model_label)
    verdict_probabilities = _build_verdict_probabilities(probs)

    print(
        f"[NLI Node] class_id={predicted_class_id}, model_label={model_label}, "
        f"verdict={label}, confidence={confidence}, "
        f"distribution={verdict_probabilities}"
    )
    return {
        "nli_label": label,
        "confidence": confidence,
        "nli_model_label": model_label,
        "nli_class_id": predicted_class_id,
        "verdict_probabilities": verdict_probabilities,
    }


def generate_motivation_node(state: HiddenState) -> dict[str, Any]:
    """Genera la risposta finale formattata citando correttamente le fonti."""
    confidence_text = _format_confidence_percentage(state["confidence"])
    sources_text = _format_sources_for_prompt(state.get("sources", []))
    probabilities_text = _format_verdict_probabilities(state.get("verdict_probabilities", {}))
    final_prompt = f"""
    Devi preparare il testo finale per un sistema di fact-checking.

    Claim dell'utente:
    \"{state['query']}\"

    Evidenze recuperate dal web:
    \"{state['researches']}\"

    Verdetto NLI:
    \"{state['nli_label']}\"

    Confidenza NLI:
    \"{confidence_text}\"

    Classe tecnica raw del modello:
    \"{state.get('nli_model_label', 'n/d')}\"

    Distribuzione delle probabilita aggregate per verdetto:
    \"{probabilities_text}\"

    Fonti candidate da citare solo se coerenti con le evidenze:
    {sources_text}

    Legenda del verdetto:
    - SUPPORTS = confermato dalle fonti
    - REFUTES = smentito dalle fonti
    - NOT ENOUGH INFO = dati insufficienti per una conclusione solida

    Scrivi in italiano una risposta professionale, chiara e logica.

    Vincoli obbligatori:
    - Non scrivere premesse meta come "ecco una proposta di risposta".
    - Apri con un titolo breve in markdown, per esempio: **Oggetto: Verifica della notizia ...**
    - Subito dopo il titolo dichiara esplicitamente il verdetto NLI esatto, senza riformularlo in senso opposto.
    - Prosegui con 3-5 paragrafi brevi e ben collegati.
    - Spiega prima il verdetto, poi le evidenze principali, poi l'eventuale causa del malinteso se emerge.
    - Devi spiegare il verdetto NLI, non ricalcolarlo e non contraddirlo.
    - Se il verdetto e' REFUTES non usare formule che implichino conferma; se e' SUPPORTS non usare formule che implichino smentita.
    - Se la confidenza non e' alta, puoi esprimere cautela ma devi restare coerente con il verdetto NLI.
    - Cita in modo sobrio 1-3 fonti, ma solo se presenti nelle fonti candidate o nelle evidenze recuperate.
    - Concludi con una frase finale netta e coerente con il verdetto.
    - Non inventare fatti non presenti nelle evidenze recuperate.
    - Mantieni un tono umano, non burocratico.
    """
    if llm is None:
        motivation_text = _fallback_motivation(state)
    else:
        res = llm.invoke(final_prompt)
        motivation_text = _prepend_technical_summary(_extract_llm_text(res), state)

    final_response = (
        motivation_text
        + "\n\n---\n**Evidenza grezza recuperata dal web (DuckDuckGo):**\n"
        + state["researches"]
    )

    if state.get("sources"):
        final_response += "\n\n**Fonti:**\n" + "\n".join(
            f"- {source}" for source in state["sources"]
        )

    return {"motivation": motivation_text, "response": final_response}


## 5. Compilazione del Grafo / Graph Compilation

**In italiano:** Costruiamo ed assembliamo la struttura del grafo. L'ordine di esecuzione lineare collegherà i quattro nodi in sequenza: estrazione e raffinamento della query -> ricerca delle evidenze sul web -> classificazione NLI (confronto premessa/ipotesi) -> formulazione del testo esplicativo finale. Compiliamo e mostriamo la visualizzazione grafica del flusso di lavoro tramite Mermaid.

**In English:** We construct and assemble the graph structure. The linear execution order connects the four nodes in sequence: query refinement -> web evidence search -> NLI classification (premise/hypothesis comparison) -> formulation of the final explanation text. We compile and display a graphical rendering of the workflow using Mermaid.


In [ ]:
from langgraph.graph import START, END, StateGraph

# Costruiamo il workflow lineare
# Build the linear workflow
workflow = StateGraph(state_schema=HiddenState, input_schema=InputState, output_schema=OutputState)

workflow.add_node("refine_query", refine_query_node)
workflow.add_node("search_web", search_node)
workflow.add_node("nli_classification", nli_classification_node)
workflow.add_node("generate_motivation", generate_motivation_node)

workflow.add_edge(START, "refine_query")
workflow.add_edge("refine_query", "search_web")
workflow.add_edge("search_web", "nli_classification")
workflow.add_edge("nli_classification", "generate_motivation")
workflow.add_edge("generate_motivation", END)

app = workflow.compile()


In [ ]:
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Errore nella generazione dell'immagine (Mermaid): {e}")
    print("Se visualizzi questo messaggio, assicurati di aver installato le dipendenze per il disegno del grafo.")


## 6. Test e Invocazione dell'Agente / Test and Agent Invocation

**In italiano:** Avviamo il grafo passando una notizia di esempio. Verrà mostrato l'output generato comprensivo del verdetto NLI, il valore di confidenza tecnica del modello, la spiegazione strutturata in italiano e i link diretti delle fonti utilizzate per la verifica del fatto.

**In English:** Launch the compiled graph by passing a sample news claim. The generated output will display the NLI verdict, the technical classification confidence score, the structured explanation in Italian, and the direct source links used to verify the claim.


In [ ]:
input_data = {"query": "Addio Bastoni, netta posizione dell’Inter: annuncio di Romano"}
result = app.invoke(input_data)

print("\n========================== RISPOSTA FINALE ==========================\n")
print(result["response"])
print("\n============================== VERDETTO ==============================\n")
print(f"Label NLI: {result['nli_label']}")
print(f"Confidenza NLI: {result['confidence']}")
if "nli_model_label" in result:
    print(f"Classe raw del modello: {result['nli_model_label']} (id={result.get('nli_class_id')})")
if "verdict_probabilities" in result:
    print(f"Probabilità verdetto: {result['verdict_probabilities']}")
print("\n=============================== FONTI ===============================\n")
for src in result.get("sources", []):
    print(f"- {src}")
